# Thử nghiệm: bỏ model touch, chỉ dùng Direction

**Câu hỏi:** Direction 3 lớp đã chứa P(không chạm) = P0, vậy lọc thêm bằng model touch có tác dụng gì?

**3 cách lọc giờ** (Direction, luật lệnh, ngưỡng trượt giữ nguyên):
| cách | lọc giờ | ngưỡng Direction tính trên |
|---|---|---|
| **touch + Direction** (hiện tại) | p_touch top x% (14 ngày) | giờ qua lọc touch (90 ngày) |
| **Direction tự lọc** | 1 − P0 của Direction top x% (14 ngày) | giờ qua lọc đó (90 ngày) |
| **chỉ Direction** | không lọc | mọi giờ (90 ngày) |

**So sánh 2 kiểu:**
- **A. Giữ nguyên tham số đã khoá**, chỉ đổi cách lọc: tác động thuần của touch.
- **B. Mỗi cách tự chọn lại tham số trên dev** (cùng lưới, ≥ `min_trades_per_year` lệnh/năm): so công bằng.

Thêm chẩn đoán: trong các tín hiệu "chỉ Direction", R theo ngũ phân vị p_touch. Nếu R tăng theo p_touch ngoài mẫu thì touch có giá trị.

**Tiêu chí:** TB seed tăng và dải seed hẹp lại (seed tệ nhất tăng).

**Cần chạy `train_strategy_h1.ipynb` trước.** Notebook này dùng lại cache fold, không train thêm.

In [1]:
import os, sys
from dataclasses import replace
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "model_training").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "model_training"))
import numpy as np
import pandas as pd
import strategy_h1 as st

pd.set_option("display.width", 220, "display.max_columns", 40)
cfg = st.load_locked("artifacts/1h_strategy_v2/config_locked.json")
OUT = Path(cfg.output_dir) / "exp_touch_gate"
OUT.mkdir(parents=True, exist_ok=True)
D = st.build(cfg)
P_dev, R_dev = st.walk_forward(D, cfg, "dev")      # từ cache
P_test, R_test = st.walk_forward(D, cfg, "test")
pd.concat([R_dev.assign(phase="dev"), R_test.assign(phase="test")])[
    ["phase", "quý", "AUC touch", "AUC touch của Direction", "AUC direction (đã chạm)"]]

Dữ liệu: 54,979 nến 1h, 53,522 mẫu (2020-08-30 -> 2026-10-08), touch 92 feature, direction 37 feature, 13s
dev 2022Q1: AUC touch 0.610 | AUC direction 0.616 | train 7,303 | 0s
dev 2022Q2: AUC touch 0.695 | AUC direction 0.536 | train 9,511 | 0s
dev 2022Q3: AUC touch 0.605 | AUC direction 0.520 | train 11,719 | 0s
dev 2022Q4: AUC touch 0.689 | AUC direction 0.601 | train 13,879 | 0s
dev 2023Q1: AUC touch 0.664 | AUC direction 0.675 | train 16,063 | 0s
dev 2023Q2: AUC touch 0.692 | AUC direction 0.588 | train 18,271 | 0s
dev 2023Q3: AUC touch 0.751 | AUC direction 0.565 | train 20,479 | 0s
dev 2023Q4: AUC touch 0.721 | AUC direction 0.566 | train 22,639 | 0s
dev 2024Q1: AUC touch 0.718 | AUC direction 0.571 | train 24,823 | 0s
dev 2024Q2: AUC touch 0.749 | AUC direction 0.587 | train 27,031 | 0s
dev 2024Q3: AUC touch 0.735 | AUC direction 0.585 | train 29,239 | 0s
dev 2024Q4: AUC touch 0.778 | AUC direction 0.626 | train 31,423 | 0s
test 2025Q1: AUC touch 0.763 | AUC direction 0.552 | tr

,phase,quý,AUC touch,AUC touch của Direction,AUC direction (đã chạm)
0,dev,2022Q1,0.610240,0.577686,0.615681
1,dev,2022Q2,0.695119,0.646283,0.536022
2,dev,2022Q3,0.605306,0.537233,0.519545
3,dev,2022Q4,0.688918,0.545331,0.600528
4,dev,2023Q1,0.664231,0.548610,0.674600
5,dev,2023Q2,0.691816,0.577534,0.588218
6,dev,2023Q3,0.751090,0.625746,0.564907
7,dev,2023Q4,0.720851,0.614273,0.565900
8,dev,2024Q1,0.718122,0.589823,0.571460
9,dev,2024Q2,0.748740,0.662858,0.586948


In [2]:
def gate(P, col):
    """Biến lọc giờ cho cả ensemble lẫn từng seed: col = p_touch | p_dtouch | None (không lọc: hằng 1.0,
    mọi giờ có đủ lịch sử 14 ngày đều qua ngưỡng)."""
    if col == "p_touch":
        return P
    Q = P.copy()
    seeds = [c[len("p_touch_s"):] for c in P.columns if c.startswith("p_touch_s")]
    Q["p_touch"] = 1.0 if col is None else P[col]
    for s in seeds:
        Q[f"p_touch_s{s}"] = 1.0 if col is None else P[f"{col}_s{s}"]
    return Q

def row(name, c, P_d, P_t):
    dev, test = st.evaluate(D, P_d, c, "dev"), st.evaluate(D, P_t, c, "test")
    S = st.seed_table(D, P_t, c, "test")
    s, sd = test["summary"], dev["summary"]
    yrs = sorted(s["trades_by_year"])
    out = {"cách": name, "touch_top": c.touch_top, "dir_top": c.dir_top, "max_pos": c.max_positions,
           "gap": c.min_gap_bars, "vào": c.entry_mode,
           "dev lệnh/năm": np.mean(list(sd["trades_by_year"].values())), "dev R/lệnh": sd["R_per_trade"],
           "test lệnh": s["trades"], "test R ens": s["R_per_trade"],
           **{f"R {y}": s["R_by_year"][y] / s["trades_by_year"][y] for y in yrs},
           "TB seed": S.loc["TB seed", "R/lệnh"], "std seed": S.loc["std seed", "R/lệnh"],
           "seed tệ nhất": S.loc["seed tệ nhất", "R/lệnh"], "thắng": s["win_rate"], "hết giờ": s["timeout_share"],
           "DD %": s.get("max_dd_pct"), "quý âm": s["negative_quarters"], "LONG": s["long_share"]}
    return out, test

GATES = {"touch + Direction": "p_touch", "Direction tự lọc": "p_dtouch", "chỉ Direction": None}

## A. Giữ nguyên tham số đã khoá, chỉ đổi cách lọc giờ

In [3]:
rows, res_A = [], {}
for name, col in GATES.items():
    c = replace(cfg, touch_top=1.0) if name == "chỉ Direction" else cfg
    r, res_A[name] = row(name, c, gate(P_dev, col), gate(P_test, col))
    rows.append(r)
A = pd.DataFrame(rows).set_index("cách")
A.to_csv(OUT / "A_same_params.csv")
A.round(3)

c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation w

,touch_top,dir_top,max_pos,gap,vào,dev lệnh/năm,dev R/lệnh,test lệnh,test R ens,R 2025,R 2026,TB seed,std seed,seed tệ nhất,thắng,hết giờ,DD %,quý âm,LONG
cách,,,,,,,,,,,,,,,,,,,
touch + Direction,0.2,0.1,5,1,limit_pullback,201.000,0.178,400,0.168,0.150,0.190,0.155,0.025,0.133,0.585,0.242,11.111,1,0.500
Direction tự lọc,0.2,0.1,5,1,limit_pullback,231.667,0.132,432,-0.016,-0.005,-0.030,-0.011,0.016,-0.029,0.468,0.347,34.179,4,0.523
chỉ Direction,1.0,0.1,5,1,limit_pullback,767.667,0.056,1486,0.038,0.036,0.042,0.035,0.009,0.022,0.491,0.406,31.077,3,0.464


## B. Mỗi cách tự chọn lại tham số trên dev

In [4]:
rows, res_B, grids = [], {}, {}
for name, col in GATES.items():
    grid = {**st.GRID, "touch_top": (1.0,)} if name == "chỉ Direction" else st.GRID
    slug = {"touch + Direction": "touch_direction", "Direction tự lọc": "direction_selfgate", "chỉ Direction": "direction_only"}[name]
    G = st.grid_search(D, gate(P_dev, col), replace(cfg, locked_from=""), grid)
    G.to_csv(OUT / f"grid_dev_{slug}.csv", index=False)
    c = st.lock_config(replace(cfg, locked_from=""), st.select_best(G), G, path=str(OUT / f"config_locked_{slug}.json"))
    r, res_B[name] = row(name, c, gate(P_dev, col), gate(P_test, col))
    rows.append(r); grids[name] = G
B = pd.DataFrame(rows).set_index("cách")
B.to_csv(OUT / "B_reselected.csv")
B.round(3)

Lưới 96 ô, 76 ô hợp lệ (>= 150 lệnh/năm), 19s


c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation w

Lưới 96 ô, 76 ô hợp lệ (>= 150 lệnh/năm), 20s


c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation w

Lưới 24 ô, 24 ô hợp lệ (>= 150 lệnh/năm), 10s


c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation w

,touch_top,dir_top,max_pos,gap,vào,dev lệnh/năm,dev R/lệnh,test lệnh,test R ens,R 2025,R 2026,TB seed,std seed,seed tệ nhất,thắng,hết giờ,DD %,quý âm,LONG
cách,,,,,,,,,,,,,,,,,,,
touch + Direction,0.2,0.10,5,1,limit_pullback,201.000,0.178,400,0.168,0.150,0.190,0.155,0.025,0.133,0.585,0.242,11.111,1,0.500
Direction tự lọc,0.2,0.10,5,2,next_open,169.333,0.161,303,-0.004,0.004,-0.013,-0.012,0.013,-0.029,0.485,0.350,22.687,3,0.535
chỉ Direction,1.0,0.05,5,1,next_open,419.667,0.080,839,0.026,0.058,-0.025,0.030,0.005,0.022,0.490,0.371,35.984,5,0.474


In [5]:
# R/lệnh test theo quý (kiểu B)
pd.DataFrame({k: v["by_quarter"]["R/lệnh"] for k, v in res_B.items()}).round(3)

,touch + Direction,Direction tự lọc,chỉ Direction
quý,,,
2025Q1,0.057,0.100,0.080
2025Q2,0.324,0.215,0.043
2025Q3,-0.001,-0.309,-0.017
2025Q4,0.211,-0.045,0.151
2026Q1,0.188,-0.083,-0.010
2026Q2,0.109,0.001,-0.001
2026Q3,0.381,0.101,-0.113
2026Q4,0.930,NaN,-0.192


## Chẩn đoán: trong các tín hiệu "chỉ Direction" (chưa giới hạn số lệnh), R theo ngũ phân vị p_touch và 1 − P0

Ngũ phân vị tính trên chính các tín hiệu đó. Nếu R tăng theo p_touch, model touch chọn được giờ Direction đúng tốt hơn.

In [6]:
c0 = replace(cfg, touch_top=1.0)
def by_quintile(P, phase):
    ev = st.evaluate(D, gate(P, None), c0, phase)
    S = ev["all_signal_trades"].drop(columns="p_touch").merge(P[P.is_test][["t_ns", "p_touch", "p_dtouch"]],
                                                              on="t_ns", how="left")
    out = []
    for col in ("p_touch", "p_dtouch"):
        q = pd.qcut(S[col], 5, labels=[1, 2, 3, 4, 5])
        g = S.groupby(q, observed=True)
        out.append(pd.DataFrame({"biến": col, "tín hiệu": g.size(), "R/lệnh": g.R.mean(), "thắng": g.R.apply(lambda r: (r > 0).mean()),
                                 "hết giờ": g.kind.apply(lambda k: (k == 3).mean()), "chạm thật": g.y_touch.mean(),
                                 "b %": g.b.mean() * 100}))
    return pd.concat(out).rename_axis("nhóm").reset_index().assign(phase=phase)

Q = pd.concat([by_quintile(P_dev, "dev"), by_quintile(P_test, "test")])
Q.to_csv(OUT / "quintiles_direction_only.csv", index=False)
display(Q.round(3))
T = P_test[P_test.is_test]
print("Spearman p_touch ~ 1-P0 trên giờ test:", round(T.p_touch.corr(T.p_dtouch, method="spearman"), 3))

c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)


,nhóm,biến,tín hiệu,R/lệnh,thắng,hết giờ,chạm thật,b %,phase
0,1,p_touch,555,-0.005,0.454,0.562,0.378,1.610,dev
1,2,p_touch,555,0.087,0.553,0.438,0.514,1.614,dev
2,3,p_touch,554,0.091,0.527,0.379,0.583,1.520,dev
3,4,p_touch,555,0.053,0.514,0.323,0.643,1.490,dev
4,5,p_touch,555,0.159,0.595,0.166,0.800,1.559,dev
5,1,p_dtouch,555,0.047,0.505,0.429,0.528,1.415,dev
6,2,p_dtouch,555,0.142,0.564,0.382,0.568,1.591,dev
7,3,p_dtouch,554,0.041,0.491,0.403,0.554,1.609,dev
8,4,p_dtouch,555,0.156,0.584,0.339,0.616,1.640,dev
9,5,p_dtouch,555,-0.002,0.499,0.315,0.652,1.538,dev


Spearman p_touch ~ 1-P0 trên giờ test: 0.538


## 3. Đánh giá theo % thay vì R (test 2025–2026)

- **% giá/lệnh** = R × khoảng SL: lợi nhuận ròng theo % giá trị lệnh nếu mọi lệnh cùng một khối lượng. Giai đoạn b lớn đóng góp nhiều hơn.
- **lãi % vốn**: rủi ro cố định 1% vốn mỗi lệnh, lãi kép (đúng cách pipeline vào lệnh). Khi đó 1R ≈ 1% vốn, b chỉ quyết định khối lượng.
- Cột seed: TB / std / seed tệ nhất của % giá/lệnh trên 5 seed.

In [7]:
import importlib
st = importlib.reload(st)      # nạp lại strategy_h1 mới (có cột % giá) nếu kernel đang giữ bản cũ
VERSIONS = [(name, replace(cfg, touch_top=1.0) if col is None else cfg, gate(P_test, col)) for name, col in GATES.items()]


def pct_row(name, c, P_t):
    ev = st.evaluate(D, P_t, c, "test")
    s, Y = ev["summary"], ev["by_year"]
    S = st.seed_table(D, P_t, c, "test").drop(index=["TB seed", "std seed", "seed tệ nhất"])
    single = S.drop(index="ens")["% giá/lệnh"]
    out = {"bản": name, "lệnh": s["trades"], "b TB %": s["b_mean_pct"], "R/lệnh": s["R_per_trade"],
           "% giá/lệnh": s["pct_per_trade"], "tổng % giá": s["total_pct"], "lãi % vốn": (s["equity_multiple"] - 1) * 100,
           "DD % vốn": s["max_dd_pct"],
           "TB seed % giá/lệnh": single.mean(), "std seed % giá/lệnh": single.std(), "seed tệ nhất % giá/lệnh": single.min()}
    for y in Y.index:
        out[f"R/lệnh {y}"], out[f"% giá/lệnh {y}"], out[f"lãi % vốn {y}"] = Y.loc[y, "R/lệnh"], Y.loc[y, "% giá/lệnh"], Y.loc[y, "lãi % vốn"]
    return out, ev

PCT = pd.DataFrame([pct_row(name, c, P_t)[0] for name, c, P_t in VERSIONS]).set_index("bản")
PCT.to_csv(OUT / "pct_compare.csv")
PCT.round(3).T

c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = t.year if freq == "Y" else t.to_period("Q").astype(str)
c:\Users\thanh\BTC_PIPELINE\model_training\strategy_h1.py:845: UserWarning: Converting to PeriodArray/Index representation w

bản,touch + Direction,Direction tự lọc,chỉ Direction
lệnh,400.000,432.000,1486.000
b TB %,1.271,1.296,1.318
R/lệnh,0.168,-0.016,0.038
% giá/lệnh,0.213,-0.039,0.048
tổng % giá,85.216,-16.769,70.861
lãi % vốn,87.978,-10.629,52.619
DD % vốn,11.111,34.179,31.077
TB seed % giá/lệnh,0.190,-0.026,0.044
std seed % giá/lệnh,0.030,0.019,0.013
seed tệ nhất % giá/lệnh,0.162,-0.049,0.028
